# Day 3 — Conversion Prediction

## Objective

Predict whether a marketing interaction will produce an order.

- `converted = 0`: no order
- `converted = 1`: order placed

The workflow is: Silver table → leakage-safe features → train/validation/test split → Logistic Regression → imbalance handling → threshold selection → final evaluation → Gold prediction table.

This notebook uses scikit-learn for model fitting because repeated Spark ML fits can overflow the 1 GB Spark Connect model cache on Databricks Serverless. Spark is still used to read the Silver table and write the Gold table.

## 1. Load the Silver table

The Silver table contains cleaned, typed, analysis-ready marketing interactions. We select only identifiers, approved input features, and the target before converting the small 10,000-row dataset to Pandas.

In [0]:
from pyspark.sql import functions as F
import numpy as np
import pandas as pd

silver_df = spark.table("silver_marketing_interactions")

categorical_features = [
    "customer_gender",
    "customer_region",
    "product_category",
    "marketing_channel",
    "device_type"
]

numeric_features = [
    "customer_age",
    "unit_price",
    "discount_percent"
]

feature_columns = categorical_features + numeric_features
target_column = "converted"

identifier_candidates = [
    "interaction_id",
    "customer_id",
    "product_id",
    "campaign_id",
    "interaction_date"
]

identifier_columns = [
    column for column in identifier_candidates
    if column in silver_df.columns
]

selected_columns = identifier_columns + feature_columns + [target_column]
pandas_df = silver_df.select(*selected_columns).toPandas()

print(f"Rows loaded: {len(pandas_df):,}")
print("Identifier columns:", identifier_columns)
display(pandas_df.head(10))

Rows loaded: 10,000
Identifier columns: ['customer_id', 'product_id', 'campaign_id', 'interaction_date']


customer_id,product_id,campaign_id,interaction_date,customer_gender,customer_region,product_category,marketing_channel,device_type,customer_age,unit_price,discount_percent,converted
C0171,P004,CMP004,2025-09-04,Male,Balochistan,Beauty,Display,Mobile,44,3000.0,0.0,0
C0989,P001,CMP003,2025-09-02,Female,Sindh,Electronics,Search,Tablet,20,50000.0,15.0,0
C0387,P004,CMP003,2025-04-07,Male,Balochistan,Beauty,Search,Desktop,32,3000.0,5.0,0
C0599,P004,CMP003,2025-02-25,Female,Sindh,Beauty,Search,Tablet,63,3000.0,20.0,0
C0873,P005,CMP002,2025-06-20,Male,Sindh,Accessories,Social Media,Desktop,24,2500.0,5.0,0
C0707,P004,CMP003,2025-08-17,Female,Punjab,Beauty,Search,Desktop,50,3000.0,10.0,0
C0034,P005,CMP002,2025-12-14,Female,Sindh,Accessories,Social Media,Mobile,59,2500.0,10.0,0
C0721,P004,CMP002,2025-11-10,Female,Punjab,Beauty,Social Media,Mobile,20,3000.0,15.0,0
C0550,P001,CMP001,2025-08-09,Male,Balochistan,Electronics,Email,Desktop,23,50000.0,25.0,0
C0686,P001,CMP002,2025-03-13,Female,Sindh,Electronics,Social Media,Desktop,39,50000.0,5.0,0


## 2. Validate the modeling data

The target is highly imbalanced: conversions are rare compared with non-conversions. Therefore, accuracy alone is misleading. We also verify that there are no missing values and that outcome columns are excluded from the input features.

Columns such as `orders`, `units_sold`, and `revenue` would leak the answer because they are known only after conversion occurs.

In [0]:
class_balance = (
    pandas_df[target_column]
    .value_counts()
    .sort_index()
    .rename_axis(target_column)
    .reset_index(name="count")
)
class_balance["percentage"] = (
    class_balance["count"] / len(pandas_df) * 100
)
display(class_balance)

missing_values = pandas_df[feature_columns + [target_column]].isna().sum()
print("Missing values:")
print(missing_values)

leakage_columns = {
    "clicks", "website_visits", "orders",
    "units_sold", "revenue", "converted"
}
input_leakage = leakage_columns.intersection(feature_columns)
print("Leakage found:", input_leakage)
assert not input_leakage, "Outcome leakage detected"
assert missing_values.sum() == 0, "Missing values must be handled"

converted,count,percentage
0,9831,98.31
1,169,1.69


Missing values:
customer_gender      0
customer_region      0
product_category     0
marketing_channel    0
device_type          0
customer_age         0
unit_price           0
discount_percent     0
converted            0
dtype: int64
Leakage found: set()


## 3. Create train, validation, and test sets

The test set is held back for one final unbiased evaluation. The validation set is taken from the training portion and is used to compare imbalance strategies and select the classification threshold.

`stratify=y` preserves approximately the same conversion rate in every split. Fixed random seeds make the results reproducible.

In [0]:
from sklearn.model_selection import train_test_split

X = pandas_df[feature_columns]
y = pandas_df[target_column].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_subtrain, X_validation, y_subtrain, y_validation = train_test_split(
    X_train, y_train,
    test_size=0.20,
    random_state=123,
    stratify=y_train
)

split_summary = pd.DataFrame({
    "dataset": ["Subtraining", "Validation", "Test"],
    "rows": [len(X_subtrain), len(X_validation), len(X_test)],
    "conversions": [y_subtrain.sum(), y_validation.sum(), y_test.sum()]
})
split_summary["conversion_rate"] = (
    split_summary["conversions"] / split_summary["rows"]
)
display(split_summary)

dataset,rows,conversions,conversion_rate
Subtraining,6400,108,0.016875
Validation,1600,27,0.016875
Test,2000,34,0.017


## 4. Build the preprocessing and Logistic Regression pipeline

Logistic Regression requires numerical inputs. `OneHotEncoder` turns each category into binary indicator columns, while `StandardScaler` places numerical features on comparable scales.

The preprocessing steps are inside the pipeline, so they are fitted only from the relevant training data. This prevents preprocessing leakage.

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

def build_pipeline(class_weight=None):
    preprocessor = ColumnTransformer([
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features
        )
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
                class_weight=class_weight
            )
        )
    ])

## 5. Evaluate baseline and class-weighted models on validation data

The baseline uses the default 0.50 threshold. Because conversions are rare, it may predict only the majority class.

The weighted model gives rare conversions more influence during training. This usually increases recall, but it can also create many false positives. These experiments use validation data, not the final test set.

In [0]:
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score,
    average_precision_score
)

def calculate_metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()

    return {
        "threshold": float(threshold),
        "true_positives": int(tp),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_negatives": int(tn),
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities)
    }

baseline_pipeline = build_pipeline(class_weight=None)
baseline_pipeline.fit(X_subtrain, y_subtrain)
baseline_validation_probabilities = baseline_pipeline.predict_proba(
    X_validation
)[:, 1]

weighted_pipeline = build_pipeline(class_weight="balanced")
weighted_pipeline.fit(X_subtrain, y_subtrain)
weighted_validation_probabilities = weighted_pipeline.predict_proba(
    X_validation
)[:, 1]

validation_comparison = pd.DataFrame([
    {
        "model": "Baseline at 0.50",
        **calculate_metrics(
            y_validation, baseline_validation_probabilities, 0.50
        )
    },
    {
        "model": "Class-weighted at 0.50",
        **calculate_metrics(
            y_validation, weighted_validation_probabilities, 0.50
        )
    }
])

display(validation_comparison)

model,threshold,true_positives,false_positives,false_negatives,true_negatives,accuracy,precision,recall,f1,roc_auc,pr_auc
Baseline at 0.50,0.5,0,0,27,1573,0.983125,0.0,0.0,0.0,0.5929222292858656,0.025931823297448503
Class-weighted at 0.50,0.5,15,600,12,973,0.6175,0.024390243902439025,0.5555555555555556,0.04672897196261682,0.5850344941254032,0.0274963494193525


## 6. Tune the baseline classification threshold

Logistic Regression produces a conversion probability. The threshold converts that probability into class 0 or 1. Reducing the threshold below 0.50 allows the model to identify rare conversions.

The threshold with the highest validation F1 score is selected. F1 balances precision and recall. Threshold selection is performed only on validation data, preventing test-set leakage.

In [0]:
threshold_values = [
    0.0025, 0.005, 0.0075, 0.01,
    0.015, 0.02, 0.03, 0.04,
    0.05, 0.075, 0.10, 0.15,
    0.20, 0.30, 0.40, 0.50
]

threshold_results = []

for threshold in threshold_values:
    metrics = calculate_metrics(
        y_validation,
        baseline_validation_probabilities,
        threshold
    )
    threshold_results.append({
        "threshold": threshold,
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "predicted_conversions": (
            metrics["true_positives"] + metrics["false_positives"]
        )
    })

threshold_results_df = (
    pd.DataFrame(threshold_results)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

best_threshold = float(threshold_results_df.loc[0, "threshold"])
display(threshold_results_df)
print(f"Selected validation threshold: {best_threshold:.4f}")

threshold,precision,recall,f1,predicted_conversions
0.05,0.05263157894736842,0.07407407407407407,0.06153846153846154,38
0.04,0.031578947368421054,0.1111111111111111,0.04918032786885246,95
0.02,0.025210084033613446,0.4444444444444444,0.04771371769383698,476
0.015,0.02247191011235955,0.5925925925925926,0.04330175913396482,712
0.03,0.023809523809523808,0.18518518518518517,0.04219409282700422,210
0.01,0.02012072434607646,0.7407407407407407,0.039177277179236046,994
0.0075,0.01948627103631532,0.8148148148148148,0.03806228373702422,1129
0.005,0.01902587519025875,0.9259259259259259,0.037285607755406416,1314
0.0025,0.017509727626459144,1.0,0.03441682600382409,1542
0.075,0.0,0.0,0.0,1


Selected validation threshold: 0.0500


## 7. Train the final model and evaluate the untouched test set

After selecting the modeling strategy and threshold, the baseline Logistic Regression is refitted using all training rows. The test set is then evaluated once.

For this imbalanced problem, precision, recall, F1, ROC-AUC, and PR-AUC are more informative than accuracy.

In [0]:
final_pipeline = build_pipeline(class_weight=None)
final_pipeline.fit(X_train, y_train)

test_probabilities = final_pipeline.predict_proba(X_test)[:, 1]
test_predictions = (
    test_probabilities >= best_threshold
).astype(int)

final_metrics = calculate_metrics(
    y_test, test_probabilities, best_threshold
)

final_results_df = pd.DataFrame([
    {"model": "Final Logistic Regression", **final_metrics}
])
display(final_results_df)

confusion_matrix_df = pd.DataFrame({
    "actual_converted": [0, 0, 1, 1],
    "predicted_converted": [0, 1, 0, 1],
    "count": [
        final_metrics["true_negatives"],
        final_metrics["false_positives"],
        final_metrics["false_negatives"],
        final_metrics["true_positives"]
    ]
})
display(confusion_matrix_df)

model,threshold,true_positives,false_positives,false_negatives,true_negatives,accuracy,precision,recall,f1,roc_auc,pr_auc
Final Logistic Regression,0.05,5,23,29,1943,0.974,0.17857142857142858,0.14705882352941177,0.16129032258064516,0.7374259469810305,0.07269291291102141


actual_converted,predicted_converted,count
0,0,1943
0,1,23
1,0,29
1,1,5


## 8. Save unbiased test predictions to the Gold table

Only test-set predictions are saved because they represent rows the final model did not see during training. The output includes the actual label for evaluation, predicted label, conversion probability, selected threshold, and model metadata.

In [0]:
prediction_output_pd = pandas_df.loc[
    X_test.index, identifier_columns + feature_columns
].copy()

prediction_output_pd.insert(
    0, "source_row_index", X_test.index.astype(int)
)
prediction_output_pd["actual_converted"] = y_test.to_numpy().astype(int)
prediction_output_pd["conversion_probability"] = (
    test_probabilities.astype(float)
)
prediction_output_pd["predicted_converted"] = (
    test_predictions.astype(int)
)
prediction_output_pd["classification_threshold"] = best_threshold
prediction_output_pd["model_name"] = "Logistic Regression"
prediction_output_pd["dataset_split"] = "test"

gold_predictions_df = (
    spark.createDataFrame(prediction_output_pd.reset_index(drop=True))
    .withColumn("prediction_timestamp", F.current_timestamp())
)

gold_predictions_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_conversion_predictions")

print("gold_conversion_predictions saved successfully.")
print(f"Saved rows: {gold_predictions_df.count():,}")

gold_conversion_predictions saved successfully.
Saved rows: 2,000


## 9. Validate the Gold table

The grouped counts below reproduce the final confusion matrix. Sorting by conversion probability also shows the interactions the model considers most likely to convert.

In [0]:
saved_predictions_df = spark.table("gold_conversion_predictions")

display(
    saved_predictions_df
    .groupBy("actual_converted", "predicted_converted")
    .count()
    .orderBy("actual_converted", "predicted_converted")
)

display(
    saved_predictions_df
    .orderBy(F.desc("conversion_probability"))
    .limit(20)
)

actual_converted,predicted_converted,count
0,0,1943
0,1,23
1,0,29
1,1,5


source_row_index,customer_id,product_id,campaign_id,interaction_date,customer_gender,customer_region,product_category,marketing_channel,device_type,customer_age,unit_price,discount_percent,actual_converted,conversion_probability,predicted_converted,classification_threshold,model_name,dataset_split,prediction_timestamp
1049,C0568,P001,CMP003,2025-10-24,Female,Punjab,Electronics,Search,Mobile,36,50000.0,30.0,0,0.06771997412446343,1,0.05,Logistic Regression,test,2026-08-27T08:40:22.394Z
1169,C0469,P002,CMP003,2025-06-21,Female,Punjab,Clothing,Search,Mobile,49,5000.0,25.0,1,0.0660914068117133,1,0.05,Logistic Regression,test,2026-08-27T08:40:22.394Z
8829,C0836,P002,CMP003,2025-06-19,Female,Punjab,Clothing,Search,Mobile,49,5000.0,25.0,0,0.0660914068117133,1,0.05,Logistic Regression,test,2026-08-27T08:40:22.394Z
1275,C0009,P005,CMP003,2025-12-17,Female,Punjab,Accessories,Search,Mobile,46,2500.0,30.0,0,0.06466010839456907,1,0.05,Logistic Regression,test,2026-08-27T08:40:22.394Z
6469,C0249,P002,CMP003,2025-02-18,Female,Sindh,Clothing,Search,Mobile,37,5000.0,30.0,0,0.06393535132546238,1,0.05,Logistic Regression,test,2026-08-27T08:40:22.394Z
7483,C0886,P002,CMP003,2025-11-14,Female,Khyber Pakhtunkhwa,Clothing,Search,Tablet,57,5000.0,30.0,1,0.06272658644912202,1,0.05,Logistic Regression,test,2026-08-27T08:40:22.394Z
2363,C0201,P002,CMP003,2025-10-16,Male,Punjab,Clothing,Search,Mobile,55,5000.0,30.0,0,0.06150630358349653,1,0.05,Logistic Regression,test,2026-08-27T08:40:22.394Z
36,C0482,P005,CMP003,2025-08-20,Female,Khyber Pakhtunkhwa,Accessories,Search,Mobile,43,2500.0,25.0,0,0.06137419604111256,1,0.05,Logistic Regression,test,2026-08-27T08:40:22.394Z
9871,C0150,P002,CMP003,2025-11-19,Female,Punjab,Clothing,Search,Mobile,20,5000.0,20.0,0,0.060771900594491106,1,0.05,Logistic Regression,test,2026-08-27T08:40:22.394Z
9464,C0266,P002,CMP003,2025-10-29,Female,Punjab,Clothing,Search,Mobile,27,5000.0,20.0,0,0.06019660795983184,1,0.05,Logistic Regression,test,2026-08-27T08:40:22.394Z


# Conclusion

The project built a leakage-safe Logistic Regression model for conversion prediction. The 0.50 default threshold was unsuitable because only about 1.69% of interactions converted. Class weighting improved conversion coverage but produced too many false positives. Validation-based threshold selection provided a better F1 balance.

In the completed run, the selected threshold was 0.05. The untouched test results were: accuracy 0.9740, precision 0.1786, recall 0.1471, F1 0.1613, ROC-AUC 0.7374, and PR-AUC 0.0727. The model found some useful ranking signal, but additional behavioral features would be needed to identify more conversions reliably.